In [ ]:
# ######fanzhuan 0.248540
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        # 创建mask：只在有效量化范围内传递梯度
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()  # 应用mask到梯度
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 一维卷积结构 (batch_size, 1, 8)
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()

        # 编码器
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, stride=1, padding=1),  # (batch, 16, 8)
            nn.BatchNorm1d(16),
            nn.LeakyReLU(inplace=True),
            nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1),  # (batch, 32, 4)
            nn.BatchNorm1d(32),
            nn.LeakyReLU(inplace=True),
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),  # (batch, 64, 2)
            nn.BatchNorm1d(64),
            nn.LeakyReLU(inplace=True),
            nn.Conv1d(64, 1, kernel_size=2, stride=1, padding=0),  # (batch, 1, 1)
            nn.Tanh()
        )

        # 解码器：使用一维卷积转置
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(1, 64, kernel_size=2, stride=1, padding=0),  # (batch, 64, 2)
            nn.BatchNorm1d(64),
            nn.LeakyReLU(inplace=True),
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # (batch, 32, 4)
            nn.BatchNorm1d(32),
            nn.LeakyReLU(inplace=True),
            nn.ConvTranspose1d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),  # (batch, 16, 8)
            nn.BatchNorm1d(16),
            nn.LeakyReLU(inplace=True),
            nn.ConvTranspose1d(16, 1, kernel_size=3, stride=1, padding=1),  # (batch, 1, 8)
            nn.Tanh()
        )

        # 4比特量化配置
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        """
        将量化后的张量转换为二进制张量。
        输入: (batch_size,) 一维张量
        输出: (batch_size, bitwidth) 二维张量
        """
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            # 从最低位到最高位提取
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        """
        将二进制张量转换回量化后的张量。
        输入: (batch_size, bitwidth) 二维张量
        输出: (batch_size,) 一维张量
        """
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        """
        将量化值解量化为原始范围的值。
        输入: (batch_size,) 一维张量
        输出: (batch_size,) 一维张量
        """
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        # 输入形状: (batch_size, 1, 8)

        # 编码：(batch_size, 1, 8) -> (batch_size, 1, 1)
        x_encoded = self.encoder(x)
        x_encoded = x_encoded * self.output_scale + self.output_bias

        # 确保量化前的张量是一维的 (batch_size,)
        x_encoded_1d = x_encoded.view(-1)

        # 量化
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)

        # 解量化
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)

        # 将一维张量恢复为解码器需要的三维形状 (batch_size, 1, 1)
        x_dequantized_3d = x_dequantized_1d.view(-1, 1, 1)

        # 解码：(batch_size, 1, 1) -> (batch_size, 1, 8)
        x_decoded = self.decoder(x_dequantized_3d)

        return x_decoded, x_quantized_1d, x_binary

# 定义数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# ==================== 数据增强部分 (仅使用翻转) ====================
# class AudioDatasetWithAugmentation(AudioDataset):
#     def __init__(self, data, flip_prob=0.5):
#         super(AudioDatasetWithAugmentation, self).__init__(data)
#         self.flip_prob = flip_prob

#     def __getitem__(self, idx):
#         sample = super().__getitem__(idx)  # 形状为 (1, 8)

#         # 随机决定是否进行翻转
#         if np.random.rand() < self.flip_prob:
#             # 对最后一个维度（特征维度）进行翻转
#             # 这相当于将 [实1, 实2, 实3, 实4, 虚1, 虚2, 虚3, 虚4]
#             # 翻转为 [虚4, 虚3, 虚2, 虚1, 实4, 实3, 实2, 实1]
#             # 注意：这种操作是否有意义取决于你的具体数据结构和任务
#             sample = torch.flip(sample, dims=[1])

#         return sample
    

class AudioDatasetWithAugmentation(AudioDataset):   #0.244457
    def __init__(self, data, flip_prob=0.5):
        super(AudioDatasetWithAugmentation, self).__init__(data)
        self.flip_prob = flip_prob

    def __getitem__(self, idx):
        sample = super().__getitem__(idx)  # 形状为 (1, 8)

        # 随机决定是否进行翻转
        if np.random.rand() < self.flip_prob:
            # --- 温和翻转 (Gentle Flip) ---
            # 1. 提取实部 (前4个元素) 并进行翻转
            # sample[0, :4] 表示第0个通道的前4个元素
            reversed_real_part = torch.flip(sample[0, :4], dims=[0])
            
            # 2. 提取虚部 (后4个元素) 并进行翻转
            # sample[0, 4:] 表示第0个通道的第4个元素及之后的所有元素
            reversed_imag_part = torch.flip(sample[0, 4:], dims=[0])
            
            # 3. 将翻转后的实部和虚部重新拼接回原样本
            sample[0, :4] = reversed_real_part
            sample[0, 4:] = reversed_imag_part
            
            # --- 操作结束 ---

        return sample


# ============================================================

# 绘制损失曲线函数
def plot_loss_curve(loss_history, title="Training Loss Curve"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(loss_history) + 1), loss_history, label='Average Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数
if __name__ == "__main__":
    # 数据路径
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"

    data = np.load(data_path)

    # 使用新的带增强的数据集类
    full_dataset = AudioDataset(data)
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size

    # 分割训练集和测试集的索引
    train_indices, test_indices = random_split(range(len(full_dataset)), [train_size, test_size])

    # 根据索引创建数据集
    train_data = data[train_indices.indices]
    test_data = data[test_indices.indices]

    # 训练集使用带翻转增强的数据，测试集不使用
    train_dataset = AudioDatasetWithAugmentation(train_data, flip_prob=0.5)
    test_dataset = AudioDataset(test_data)

    # 创建DataLoader
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # 创建模型
    codec = CodecSystem()

    # 检查是否有可用的GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    print(f"使用设备: {device}")

    # 损失函数和优化器
    criterion = nn.L1Loss()
    optimizer = optim.AdamW(codec.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2,
                                                                eta_min=0.00005)  # 50/30

    # 训练循环
    num_epochs = 100
    codec.train()

    loss_history = []

    print("\n开始训练...")
    for epoch in range(num_epochs):
        running_loss = 0.0

        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)

            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)

            optimizer.step()

            running_loss += loss.item()

            if (i + 1) % 100 == 0:
                print(
                    f'Epoch [{epoch + 1}/{num_epochs}], Batch [{i + 1}/{len(train_loader)}], Loss: {loss.item():.6f}')

        scheduler.step()

        epoch_loss = running_loss / len(train_loader)
        loss_history.append(epoch_loss)
        current_lr = scheduler.get_last_lr()[0]
        print(f'Epoch [{epoch + 1}/{num_epochs}], Average Loss: {epoch_loss:.6f}, LR: {current_lr:.6f}')

    plot_loss_curve(loss_history, title="Training Loss Curve (L1 Loss)")

    # 测试阶段
    print("\n开始测试...")
    codec.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_loss += loss.item() * batch_data.size(0)

    average_test_loss = total_loss / len(test_dataset)
    print(f"测试集平均L1 Loss: {average_test_loss:.6f}")






In [ ]:
######  幅度缩放＋高斯噪声(guonihe maybe) 0.238814
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        # 创建mask：只在有效量化范围内传递梯度
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()  # 应用mask到梯度
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 一维卷积结构 (batch_size, 1, 8)
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()

        # 编码器
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, stride=1, padding=1),  # (batch, 16, 8)
            nn.BatchNorm1d(16),
            nn.LeakyReLU(inplace=True),
            nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1),  # (batch, 32, 4)
            nn.BatchNorm1d(32),
            nn.LeakyReLU(inplace=True),
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),  # (batch, 64, 2)
            nn.BatchNorm1d(64),
            nn.LeakyReLU(inplace=True),
            nn.Conv1d(64, 1, kernel_size=2, stride=1, padding=0),  # (batch, 1, 1)
            nn.Tanh()
        )

        # 解码器：使用一维卷积转置
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(1, 64, kernel_size=2, stride=1, padding=0),  # (batch, 64, 2)
            nn.BatchNorm1d(64),
            nn.LeakyReLU(inplace=True),
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # (batch, 32, 4)
            nn.BatchNorm1d(32),
            nn.LeakyReLU(inplace=True),
            nn.ConvTranspose1d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),  # (batch, 16, 8)
            nn.BatchNorm1d(16),
            nn.LeakyReLU(inplace=True),
            nn.ConvTranspose1d(16, 1, kernel_size=3, stride=1, padding=1),  # (batch, 1, 8)
            nn.Tanh()
        )

        # 4比特量化配置
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        """
        将量化后的张量转换为二进制张量。
        输入: (batch_size,) 一维张量
        输出: (batch_size, bitwidth) 二维张量
        """
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            # 从最低位到最高位提取
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        """
        将二进制张量转换回量化后的张量。
        输入: (batch_size, bitwidth) 二维张量
        输出: (batch_size,) 一维张量
        """
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        """
        将量化值解量化为原始范围的值。
        输入: (batch_size,) 一维张量
        输出: (batch_size,) 一维张量
        """
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        # 输入形状: (batch_size, 1, 8)

        # 编码：(batch_size, 1, 8) -> (batch_size, 1, 1)
        x_encoded = self.encoder(x)
        x_encoded = x_encoded * self.output_scale + self.output_bias

        # 确保量化前的张量是一维的 (batch_size,)
        x_encoded_1d = x_encoded.view(-1)

        # 量化
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)

        # 解量化
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)

        # 将一维张量恢复为解码器需要的三维形状 (batch_size, 1, 1)
        x_dequantized_3d = x_dequantized_1d.view(-1, 1, 1)

        # 解码：(batch_size, 1, 1) -> (batch_size, 1, 8)
        x_decoded = self.decoder(x_dequantized_3d)

        return x_decoded, x_quantized_1d, x_binary

# 定义数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

# ==================== 新增的数据增强部分 ====================
class AudioDatasetWithAugmentation(AudioDataset):
    def __init__(self, data, augment_prob=0.5):
        super(AudioDatasetWithAugmentation, self).__init__(data)
        self.augment_prob = augment_prob

    def __getitem__(self, idx):
        sample = super().__getitem__(idx)  # (1, 8)
     
     ###  虚部符号翻转 
        # if np.random.rand() < 0.3:  # 30% 的概率翻转虚部符号
        #     sample[0, 4:] = -sample[0, 4:]  # 翻转后4个元素（虚部）的符号
        
        # # # 随机决定是否进行增强
        # if np.random.rand() < self.augment_prob:
        #     # 1. 幅度缩放 (对实部和虚部同时缩放)
        #     # 缩放因子在 [0.8, 1.2] 之间
        #     scale_factor = np.random.uniform(0.8, 1.2)
        #     sample = sample * scale_factor

        #     #2. 加性噪声注入
        #     #噪声强度为数据标准差的 0.01 到 0.05 倍
        #     noise_std = np.random.uniform(0.01, 0.05) * sample.std()
        #     noise = torch.randn_like(sample) * noise_std
        #     sample = sample + noise
        
        ######实部虚部独立缩放
        # if np.random.rand() < 0.4: 
        #      scale_real = np.random.uniform(0.9, 1.1)
        #      scale_imag = np.random.uniform(0.9, 1.1)
        #      sample[0, :4] *= scale_real  # 实部独立缩放
        #      sample[0, 4:] *= scale_imag  # 虚部独立缩放

        
        ###循环移位  buhao
        if np.random.rand() < 0.2:
                shift = np.random.randint(1, 4)
                sample[0, :4] = torch.roll(sample[0, :4], shifts=shift, dims=0)
                sample[0, 4:] = torch.roll(sample[0, 4:], shifts=shift, dims=0)


        return sample
# ============================================================

# 绘制损失曲线函数
def plot_loss_curve(loss_history, title="Training Loss Curve"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(loss_history) + 1), loss_history, label='Average Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数
if __name__ == "__main__":
    # 数据路径
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"

    data = np.load(data_path)


    ###quchong    buhao
    # data_flat = data.reshape(data.shape[0], -1)
    # _, unique_indices = np.unique(data_flat, axis=0, return_index=True)
    # data = data[np.sort(unique_indices)]


    # 使用新的带增强的数据集类来创建训练集
    # 注意：测试集通常不进行数据增强，以评估真实泛化能力
    full_dataset = AudioDataset(data)
    train_size = int(0.8 * len(full_dataset))
    test_size = len(full_dataset) - train_size

    # 从原始数据集中分割出训练和测试的索引
    train_indices, test_indices = random_split(range(len(full_dataset)), [train_size, test_size])

    # 根据索引创建带增强的训练集和不带增强的测试集
    train_data = data[train_indices.indices]
    test_data = data[test_indices.indices]

    train_dataset = AudioDatasetWithAugmentation(train_data, augment_prob=0.5)
    test_dataset = AudioDataset(test_data)

    # 创建DataLoader
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # 创建模型
    codec = CodecSystem()

    # 检查是否有可用的GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    print(f"使用设备: {device}")

    # 损失函数和优化器
    criterion = nn.L1Loss()
    optimizer = optim.AdamW(codec.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2,
                                                                eta_min=0.00005)  # 50/30

    # 训练循环
    num_epochs = 100
    codec.train()

    loss_history = []

    print("\n开始训练...")
    for epoch in range(num_epochs):
        running_loss = 0.0

        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)

            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)

            optimizer.step()

            running_loss += loss.item()

            if (i + 1) % 100 == 0:
                print(
                    f'Epoch [{epoch + 1}/{num_epochs}], Batch [{i + 1}/{len(train_loader)}], Loss: {loss.item():.6f}')

        scheduler.step()

        epoch_loss = running_loss / len(train_loader)
        loss_history.append(epoch_loss)
        current_lr = scheduler.get_last_lr()[0]
        print(f'Epoch [{epoch + 1}/{num_epochs}], Average Loss: {epoch_loss:.6f}, LR: {current_lr:.6f}')

    plot_loss_curve(loss_history, title="Training Loss Curve (L1 Loss)")

    # 测试阶段
    print("\n开始测试...")
    codec.eval()
    total_loss = 0.0

    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_loss += loss.item() * batch_data.size(0)

    average_test_loss = total_loss / len(test_dataset)
    print(f"测试集平均L1 Loss: {average_test_loss:.6f}")





In [ ]:
#####添加数据预处理 0.249822
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt  

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        # 创建mask：只在有效量化范围内传递梯度
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()  # 应用mask到梯度
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 一维卷积结构 (batch_size, 1, 8)
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        
        # 编码器
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, stride=1, padding=1),  # (batch, 16, 8)
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1),  # (batch, 32, 4)
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),  # (batch, 64, 2)
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 1, kernel_size=2, stride=1, padding=0),  # (batch, 1, 1)
            nn.Tanh()
        )
        
        # 解码器：使用一维卷积转置
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(1, 64, kernel_size=2, stride=1, padding=0),  # (batch, 64, 2)
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # (batch, 32, 4)
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),  # (batch, 16, 8)
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(16, 1, kernel_size=3, stride=1, padding=1),  # (batch, 1, 8)
            nn.Tanh()
        )
        
        # 4比特量化配置
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        """
        将量化后的张量转换为二进制张量。
        输入: (batch_size,) 一维张量
        输出: (batch_size, bitwidth) 二维张量
        """
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            # 从最低位到最高位提取
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        """
        将二进制张量转换回量化后的张量。
        输入: (batch_size, bitwidth) 二维张量
        输出: (batch_size,) 一维张量
        """
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        """
        将量化值解量化为原始范围的值。
        输入: (batch_size,) 一维张量
        输出: (batch_size,) 一维张量
        """
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        # 输入形状: (batch_size, 1, 8)
        
        # 编码：(batch_size, 1, 8) -> (batch_size, 1, 1)
        x_encoded = self.encoder(x)  
        x_encoded = x_encoded * self.output_scale + self.output_bias
        
        # 确保量化前的张量是一维的 (batch_size,)
        # .view(-1) 会将任何形状（如 (batch_size, 1, 1) 或 (batch_size, 1)）安全地展平为一维
        x_encoded_1d = x_encoded.view(-1)
        
        # 量化
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)
        
        # 解量化
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)
        
        # 将一维张量恢复为解码器需要的三维形状 (batch_size, 1, 1)
        x_dequantized_3d = x_dequantized_1d.view(-1, 1, 1)
        
        # 解码：(batch_size, 1, 1) -> (batch_size, 1, 8)
        x_decoded = self.decoder(x_dequantized_3d)
        
        return x_decoded, x_quantized_1d, x_binary

# 定义数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线函数
def plot_loss_curve(loss_history, title="Training Loss Curve"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(loss_history) + 1), loss_history, label='Average Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数
if __name__ == "__main__":
    # 数据路径
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    
    # 加载数据
    data = np.load(data_path)

    # --- 核心修改：添加数据预处理 ---
    print("开始进行数据预处理...")
    # 1. 计算数据的全局最大绝对值
    max_abs_value = np.max(np.abs(data))
    print(f"数据的全局最大绝对值为: {max_abs_value:.6f}")
    
    # 2. 归一化到 [-1, 1] 范围
    if max_abs_value > 0:  # 避免除以零的错误
        data = data / max_abs_value
        print("数据已成功归一化到 [-1, 1] 范围。")
    else:
        print("警告：数据的所有值均为0，无需进行归一化。")
    # --- 数据预处理结束 ---

    dataset = AudioDataset(data)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    # 创建DataLoader
    batch_size = 128
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    
    # 创建模型
    codec = CodecSystem()
    
    # 检查是否有可用的GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    print(f"使用设备: {device}")
    
    # 损失函数和优化器
    criterion = nn.L1Loss()
    optimizer = optim.AdamW(codec.parameters(),lr=1e-3,weight_decay=1e-4 )
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50,T_mult=2,eta_min=0.00005) #50/30
    
    # 训练循环
    num_epochs = 100
    codec.train()
    
    loss_history = []
    
    print("\n开始训练...")
    for epoch in range(num_epochs):
        running_loss = 0.0
        
        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)
            
            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            running_loss += loss.item()
            
            if (i + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.6f}')
        
        scheduler.step()
        
        epoch_loss = running_loss / len(train_loader)
        loss_history.append(epoch_loss)
        current_lr = scheduler.get_last_lr()[0]
        print(f'Epoch [{epoch+1}/{num_epochs}], Average Loss: {epoch_loss:.6f}, LR: {current_lr:.6f}')
    
    plot_loss_curve(loss_history, title="Training Loss Curve (L1 Loss)")
    
    # 测试阶段
    print("\n开始测试...")
    codec.eval()
    total_loss = 0.0
    
    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_loss += loss.item() * batch_data.size(0)
    
    average_test_loss = total_loss / len(test_dataset)
    print(f"测试集平均L1 Loss: {average_test_loss:.6f}")





In [ ]:
######2 添加早停机制 0.239971
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 一维卷积编解码器
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        # 编码器
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 1, kernel_size=2, stride=1, padding=0),
            nn.Tanh()
        )
        # 解码器
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(1, 64, kernel_size=2, stride=1, padding=0),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(16, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )
        # 量化
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)
        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        x_encoded = self.encoder(x)
        x_encoded = x_encoded * self.output_scale + self.output_bias
        x_encoded_1d = x_encoded.view(-1)
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)
        x_dequantized_3d = x_dequantized_1d.view(-1, 1, 1)
        x_decoded = self.decoder(x_dequantized_3d)
        return x_decoded, x_quantized_1d, x_binary

# 数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线
def plot_loss_curve(train_history, val_history, title="Training and Validation Loss"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_history) + 1), train_history, label='Train Loss')
    plt.plot(range(1, len(val_history) + 1), val_history, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数
if __name__ == "__main__":
    # 数据加载
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    data = np.load(data_path)
    dataset = AudioDataset(data)

    # 划分训练集、验证集和测试集
    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

    # DataLoader
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # 模型初始化
    codec = CodecSystem()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    print(f"使用设备: {device}")

    # 优化器和调度器
    criterion = nn.L1Loss()
    optimizer = optim.AdamW(
        codec.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=30,
        T_mult=2,
        eta_min=1e-5
    )

    # 早停机制
    patience = 15
    best_val_loss = float('inf')
    counter = 0
    best_model_state = None  

    # 训练循环
    num_epochs = 100
    train_loss_history = []
    val_loss_history = []

    print("\n开始训练...")
    for epoch in range(num_epochs):
        # 训练阶段
        running_train_loss = 0.0
        codec.train()
        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)
            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
            optimizer.step()
            running_train_loss += loss.item()

            if (i + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Train Loss: {loss.item():.6f}')

        epoch_train_loss = running_train_loss / len(train_loader)
        train_loss_history.append(epoch_train_loss)

        # 验证阶段
        running_val_loss = 0.0
        codec.eval()
        with torch.no_grad():
            for batch_data in val_loader:
                batch_data = batch_data.to(device)
                outputs, _, _ = codec(batch_data)
                loss = criterion(outputs, batch_data)
                running_val_loss += loss.item() * batch_data.size(0)

        epoch_val_loss = running_val_loss / len(val_dataset)
        val_loss_history.append(epoch_val_loss)
        
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_train_loss:.6f}, Val Loss: {epoch_val_loss:.6f}, LR: {current_lr:.6f}')

        # 早停逻辑（仅在内存中保存最佳状态）
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            counter = 0
            best_model_state = codec.state_dict()  # 保存到内存
            print(f'Validation loss improved! Best model state saved in memory.')
        else:
            counter += 1
            print(f'Validation loss did not improve for {counter} epochs.')
            if counter >= patience:
                print(f"早停触发！在第 {epoch+1} 个epoch停止训练。")
                break
    
    # 绘制损失曲线
    plot_loss_curve(train_loss_history, val_loss_history)
    
    # 最终测试（使用内存中的最佳模型状态）
    print("\n开始最终测试...")
    if best_model_state is not None:
        codec.load_state_dict(best_model_state)  # 加载最佳状态
    codec.eval()
    total_test_loss = 0.0
    
    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_test_loss += loss.item() * batch_data.size(0)
    
    average_test_loss = total_test_loss / len(test_dataset)
    print(f"测试集平均L1 Loss: {average_test_loss:.6f}")

    


In [ ]:
#####添加数据预处理 0.250097

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
import os

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 一维卷积编解码器
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        # 编码器
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 1, kernel_size=2, stride=1, padding=0),
            nn.Tanh()
        )
        # 解码器
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(1, 64, kernel_size=2, stride=1, padding=0),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(16, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )
        # 量化
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)
        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        x_encoded = self.encoder(x)
        x_encoded = x_encoded * self.output_scale + self.output_bias
        x_encoded_1d = x_encoded.view(-1)
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)
        x_dequantized_3d = x_dequantized_1d.view(-1, 1, 1)
        x_decoded = self.decoder(x_dequantized_3d)
        return x_decoded, x_quantized_1d, x_binary

# 数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线
def plot_loss_curve(train_history, val_history, title="Training and Validation Loss"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_history) + 1), train_history, label='Train Loss')
    plt.plot(range(1, len(val_history) + 1), val_history, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数
if __name__ == "__main__":
    # 数据加载
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    if not os.path.exists(data_path):
        raise FileNotFoundError(f"数据文件未找到: {data_path}")
        
    print(f"正在从 {data_path} 加载数据...")
    data = np.load(data_path)
    
    # --- 核心修改：添加数据预处理 ---
    print("开始进行数据预处理（归一化到 [-1, 1]）...")
    # 计算所有数据的最大绝对值，用于归一化
    max_abs_value = np.max(np.abs(data))
    if max_abs_value > 0:
        data = data / max_abs_value
        print(f"数据预处理完成。全局最大绝对值为: {max_abs_value:.6f}")
    else:
        print("警告：数据的所有值均为0，无需进行归一化。")
    
    dataset = AudioDataset(data)

    # 划分训练集、验证集和测试集
    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])
    print(f"数据集划分完成: 训练集 {len(train_dataset)} 个样本, 验证集 {len(val_dataset)} 个样本, 测试集 {len(test_dataset)} 个样本。")

    # DataLoader
    batch_size = 256
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

    # 模型初始化
    codec = CodecSystem()
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    codec.to(device)
    print(f"\n使用设备: {device}")

    # 优化器和调度器
    criterion = nn.L1Loss()
    optimizer = optim.AdamW(
        codec.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=30,
        T_mult=2,
        eta_min=1e-5
    )

    # 早停机制
    patience = 15
    best_val_loss = float('inf')
    counter = 0
    best_model_state = None  

    # 训练循环
    num_epochs = 100
    train_loss_history = []
    val_loss_history = []

    print("\n开始训练...")
    for epoch in range(num_epochs):
        # 训练阶段
        running_train_loss = 0.0
        codec.train()
        for i, batch_data in enumerate(train_loader):
            batch_data = batch_data.to(device)
            optimizer.zero_grad()
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
            optimizer.step()
            running_train_loss += loss.item()

            if (i + 1) % 100 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Train Loss: {loss.item():.6f}')

        epoch_train_loss = running_train_loss / len(train_loader)
        train_loss_history.append(epoch_train_loss)

        # 验证阶段
        running_val_loss = 0.0
        codec.eval()
        with torch.no_grad():
            for batch_data in val_loader:
                batch_data = batch_data.to(device)
                outputs, _, _ = codec(batch_data)
                loss = criterion(outputs, batch_data)
                running_val_loss += loss.item() * batch_data.size(0)

        epoch_val_loss = running_val_loss / len(val_dataset)
        val_loss_history.append(epoch_val_loss)
        
        scheduler.step()
        current_lr = scheduler.get_last_lr()[0]

        print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_train_loss:.6f}, Val Loss: {epoch_val_loss:.6f}, LR: {current_lr:.6f}')

        # 早停逻辑
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            counter = 0
            best_model_state = codec.state_dict()
            print(f'Validation loss improved! Best model state saved in memory.')
        else:
            counter += 1
            print(f'Validation loss did not improve for {counter} epochs. Patience: {counter}/{patience}')
            if counter >= patience:
                print(f"早停触发！在第 {epoch+1} 个epoch停止训练。")
                break
    
    # 绘制损失曲线
    plot_loss_curve(train_loss_history, val_loss_history)
    
    # 最终测试
    print("\n开始最终测试...")
    if best_model_state is not None:
        codec.load_state_dict(best_model_state)
    codec.eval()
    total_test_loss = 0.0
    
    with torch.no_grad():
        for batch_data in test_loader:
            batch_data = batch_data.to(device)
            outputs, _, _ = codec(batch_data)
            loss = criterion(outputs, batch_data)
            total_test_loss += loss.item() * batch_data.size(0)
    
    average_test_loss = total_test_loss / len(test_dataset)
    print(f"测试集平均L1 Loss: {average_test_loss:.6f}")




In [ ]:

#####在1的基础上添加网格搜索法
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import os
import matplotlib.pyplot as plt
import itertools  # 导入itertools用于生成超参数组合

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 一维卷积结构 (batch_size, 1, 8)
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        
        # 编码器
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 1, kernel_size=2, stride=1, padding=0),
            nn.Tanh()
        )
        
        # 解码器
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(1, 64, kernel_size=2, stride=1, padding=0),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(16, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )
        
        # 4比特量化配置
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)

        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        x_encoded = self.encoder(x)
        x_encoded = x_encoded * self.output_scale + self.output_bias
        x_encoded_1d = x_encoded.view(-1)
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)
        x_dequantized_3d = x_dequantized_1d.view(-1, 1, 1)
        x_decoded = self.decoder(x_dequantized_3d)
        return x_decoded, x_quantized_1d, x_binary

# 定义数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线函数 (在网格搜索中，我们主要关注最终结果，此函数可用于单独分析最佳模型)
def plot_loss_curve(loss_history, title="Training Loss Curve"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(loss_history) + 1), loss_history, label='Average Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数
if __name__ == "__main__":
    # 数据路径
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    
    data = np.load(data_path)
    dataset = AudioDataset(data)
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
    
    # --- 定义超参数网格 (组合1) ---
    param_grid = {
        'lr': [5e-4, 1e-3, 2e-3],
        'batch_size': [64, 128],
        'weight_decay': [1e-4, 5e-4],
        'T_0': [30, 50],
        'T_mult': [1, 2],
        'eta_min': [1e-5, 5e-5]
    }
    num_epochs = 150 # 组合1的总epoch数
    
    # 生成所有超参数组合
    param_combinations = list(itertools.product(
        param_grid['lr'],
        param_grid['batch_size'],
        param_grid['weight_decay'],
        param_grid['T_0'],
        param_grid['T_mult'],
        param_grid['eta_min']
    ))
    
    print(f"总共有 {len(param_combinations)} 种超参数组合需要训练。")
    
    # 检查是否有可用的GPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")

    # 记录最佳结果
    best_val_loss = float('inf')
    best_hyperparams = None
    
    # --- 遍历所有超参数组合 ---
    for idx, (lr, batch_size, weight_decay, T_0, T_mult, eta_min) in enumerate(param_combinations, 1):
        print(f"\n--- 正在训练第 {idx}/{len(param_combinations)} 组超参数 ---")
        print(f"当前组合: lr={lr:.6f}, batch_size={batch_size}, weight_decay={weight_decay:.6f}, T_0={T_0}, T_mult={T_mult}, eta_min={eta_min:.6f}")

        # 根据当前batch_size创建DataLoader
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
        
        # 创建新的模型实例
        codec = CodecSystem()
        codec.to(device)
        
        # 定义损失函数、优化器和调度器
        criterion = nn.L1Loss()
        optimizer = optim.AdamW(codec.parameters(), lr=lr, weight_decay=weight_decay)
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=T_0, T_mult=T_mult, eta_min=eta_min)
        
        # 训练循环
        codec.train()
        loss_history = []
        
        for epoch in range(num_epochs):
            running_loss = 0.0
            
            for i, batch_data in enumerate(train_loader):
                batch_data = batch_data.to(device)
                
                optimizer.zero_grad()
                outputs, _, _ = codec(batch_data)
                loss = criterion(outputs, batch_data)
                loss.backward()
                
                torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
                
                optimizer.step()
                
                running_loss += loss.item()
                
                # 每100个batch打印一次信息
                if (i + 1) % 100 == 0:
                    print(f'  Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Loss: {loss.item():.6f}')
            
            scheduler.step()
            
            epoch_loss = running_loss / len(train_loader)
            loss_history.append(epoch_loss)
            current_lr = scheduler.get_last_lr()[0]
            print(f'  Epoch [{epoch+1}/{num_epochs}], Average Loss: {epoch_loss:.6f}, LR: {current_lr:.6f}')

        # 验证阶段
        print("  开始验证...")
        codec.eval()
        total_loss = 0.0
        
        with torch.no_grad():
            for batch_data in test_loader:
                batch_data = batch_data.to(device)
                outputs, _, _ = codec(batch_data)
                loss = criterion(outputs, batch_data)
                total_loss += loss.item() * batch_data.size(0)
        
        average_val_loss = total_loss / len(test_dataset)
        print(f'  验证集平均L1 Loss: {average_val_loss:.6f}')

        # 更新最佳结果
        if average_val_loss < best_val_loss:
            print(f'  *** 发现更优模型! 验证损失从 {best_val_loss:.6f} 降至 {average_val_loss:.6f} ***')
            best_val_loss = average_val_loss
            best_hyperparams = {
                'lr': lr,
                'batch_size': batch_size,
                'weight_decay': weight_decay,
                'T_0': T_0,
                'T_mult': T_mult,
                'eta_min': eta_min,
                'val_loss': average_val_loss
            }
            
            # 保存最佳模型（可选）
            # torch.save(codec.state_dict(), 'best_model.pth')
            
    # 打印最终最佳结果
    print("\n" + "="*50)
    print("网格搜索完成!")
    print("="*50)
    if best_hyperparams:
        print("最佳超参数组合:")
        for key, value in best_hyperparams.items():
            print(f"  {key}: {value}")
        print(f"\n对应的最佳验证损失: {best_val_loss:.6f}")
    else:
        print("未找到有效的超参数组合。")



In [ ]:

######1+早停＋超参数
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np
import matplotlib.pyplot as plt
import itertools  

# 设置随机种子
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

# 定义量化函数
class QuantizeSTEFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, bitwidth, quant_min, quant_max):
        levels = 2 ** bitwidth
        scale = (quant_max - quant_min) / (levels - 1)
        x_normalized = (x - quant_min) / scale
        ctx.save_for_backward(x_normalized)
        ctx.levels = levels
        x_quantized = torch.round(torch.clamp(x_normalized, 0, levels - 1))
        return x_quantized.long().squeeze()

    @staticmethod
    def backward(ctx, grad_output):
        x_normalized, = ctx.saved_tensors
        levels = ctx.levels
        mask = (x_normalized >= 0) & (x_normalized <= levels - 1)
        grad_input = grad_output * mask.float()
        return grad_input, None, None, None

class QuantizeSTELayer(nn.Module):
    def __init__(self, bitwidth, quant_min, quant_max):
        super(QuantizeSTELayer, self).__init__()
        self.bitwidth = bitwidth
        self.quant_min = quant_min
        self.quant_max = quant_max

    def forward(self, x):
        return QuantizeSTEFunction.apply(x, self.bitwidth, self.quant_min, self.quant_max)

# 一维卷积编解码器
class CodecSystem(nn.Module):
    def __init__(self):
        super(CodecSystem, self).__init__()
        # 编码器
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.Conv1d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.Conv1d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 1, kernel_size=2, stride=1, padding=0),
            nn.Tanh()
        )
        # 解码器
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(1, 64, kernel_size=2, stride=1, padding=0),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.ConvTranspose1d(16, 1, kernel_size=3, stride=1, padding=1),
            nn.Tanh()
        )
        # 量化
        self.bitwidth = 4
        self.levels = 2 ** self.bitwidth
        self.quant_min = -2
        self.quant_max = 2
        self.quantizer = QuantizeSTELayer(self.bitwidth, self.quant_min, self.quant_max)
        self.output_scale = nn.Parameter(torch.tensor(1.0))
        self.output_bias = nn.Parameter(torch.tensor(0.0))

    def quantized_to_binary(self, quantized):
        batch_size = quantized.numel()
        binary = torch.zeros(batch_size, self.bitwidth, dtype=torch.int8, device=quantized.device)
        for i in range(self.bitwidth):
            binary[:, self.bitwidth - 1 - i] = (quantized >> i) & 1
        return binary

    def binary_to_quantized(self, binary):
        batch_size = binary.shape[0]
        quantized = torch.zeros(batch_size, dtype=torch.long, device=binary.device)
        for i in range(self.bitwidth):
            quantized = quantized * 2 + binary[:, i]
        return quantized

    def dequantize(self, x_quantized):
        scale = (self.quant_max - self.quant_min) / (self.levels - 1)
        return x_quantized.float() * scale + self.quant_min

    def forward(self, x):
        x_encoded = self.encoder(x)
        x_encoded = x_encoded * self.output_scale + self.output_bias
        x_encoded_1d = x_encoded.view(-1)
        x_quantized_1d = self.quantizer(x_encoded_1d)
        x_binary = self.quantized_to_binary(x_quantized_1d)
        x_quantized_recovered_1d = self.binary_to_quantized(x_binary)
        x_dequantized_1d = self.dequantize(x_quantized_recovered_1d)
        x_dequantized_3d = x_dequantized_1d.view(-1, 1, 1)
        x_decoded = self.decoder(x_dequantized_3d)
        return x_decoded, x_quantized_1d, x_binary

# 数据集类
class AudioDataset(Dataset):
    def __init__(self, data):
        self.data = torch.from_numpy(data).float()
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

# 绘制损失曲线 
def plot_loss_curve(train_history, val_history, title="Training and Validation Loss"):
    plt.figure(figsize=(10, 6))
    plt.plot(range(1, len(train_history) + 1), train_history, label='Train Loss')
    plt.plot(range(1, len(val_history) + 1), val_history, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(title)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

# 主函数
if __name__ == "__main__":
    
    data_path = "/home/mengyao.li/project/compression/data/processed_v_wideband.npy"
    data = np.load(data_path)
    dataset = AudioDataset(data)

    
    train_size = int(0.8 * len(dataset))
    val_size = int(0.1 * len(dataset))
    test_size = len(dataset) - train_size - val_size
    train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

    # 
    param_grid = {
        'lr': [5e-4, 1e-3, 2e-3],
        'batch_size': [128, 256],
        'weight_decay': [1e-4, 5e-4],
        'T_0': [30, 50],
        'T_mult': [1, 2],
        'eta_min': [1e-5, 5e-5]

    }
    # 生成所有超参数组合
    param_combinations = list(itertools.product(
        param_grid['lr'],
        param_grid['batch_size'],
        param_grid['weight_decay'],
        param_grid['T_0'],
        param_grid['T_mult'],
        param_grid['eta_min']
    ))
    
    print(f"总共有 {len(param_combinations)} 种超参数组合需要训练。")
    

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"使用设备: {device}")

    # 记录全局最佳结果
    global_best_val_loss = float('inf')
    global_best_hyperparams = None
    global_best_model_state = None

    #遍历所有超参数组合 
    for combo_idx, (lr, batch_size, weight_decay, T_0, T_mult, eta_min) in enumerate(param_combinations, 1):
        print("\n" + "="*60)
        print(f"--- 正在训练第 {combo_idx}/{len(param_combinations)} 组超参数 ---")
        print(f"当前组合: lr={lr:.6f}, batch_size={batch_size}, weight_decay={weight_decay:.6f}, T_0={T_0}, T_mult={T_mult}, eta_min={eta_min:.6f}")
        print("="*60)

        # 根据当前batch_size创建DataLoader
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

        # 创建新的模型实例
        codec = CodecSystem()
        codec.to(device)

        # 定义优化器和调度器
        criterion = nn.L1Loss()
        optimizer = optim.AdamW(
            codec.parameters(),
            lr=lr,
            weight_decay=weight_decay
        )
        scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer,
            T_0=T_0,
            T_mult=T_mult,
            eta_min=eta_min
        )

        # 早停机制 (每组参数独立)
        patience = 15
        best_val_loss = float('inf')
        counter = 0
        best_model_state = None

        # 训练循环
        num_epochs = 100 
        train_loss_history = []
        val_loss_history = []

        for epoch in range(num_epochs):
            # 训练阶段
            running_train_loss = 0.0
            codec.train()
            for i, batch_data in enumerate(train_loader):
                batch_data = batch_data.to(device)
                optimizer.zero_grad()
                outputs, _, _ = codec(batch_data)
                loss = criterion(outputs, batch_data)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(codec.parameters(), max_norm=1.0)
                optimizer.step()
                running_train_loss += loss.item()

                if (i + 1) % 100 == 0:
                    print(f'Epoch [{epoch+1}/{num_epochs}], Batch [{i+1}/{len(train_loader)}], Train Loss: {loss.item():.6f}')

            epoch_train_loss = running_train_loss / len(train_loader)
            train_loss_history.append(epoch_train_loss)

            # 验证阶段
            running_val_loss = 0.0
            codec.eval()
            with torch.no_grad():
                for batch_data in val_loader:
                    batch_data = batch_data.to(device)
                    outputs, _, _ = codec(batch_data)
                    loss = criterion(outputs, batch_data)
                    running_val_loss += loss.item() * batch_data.size(0) # 注意：这里用batch_size加权

            epoch_val_loss = running_val_loss / len(val_dataset)
            val_loss_history.append(epoch_val_loss)
            
            scheduler.step()
            current_lr = scheduler.get_last_lr()[0]

            print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {epoch_train_loss:.6f}, Val Loss: {epoch_val_loss:.6f}, LR: {current_lr:.6f}')

            # 早停逻辑
            if epoch_val_loss < best_val_loss:
                best_val_loss = epoch_val_loss
                counter = 0
                best_model_state = codec.state_dict()
                print(f'  -> 验证损失改善! 最佳模型状态已保存。')
            else:
                counter += 1
                print(f'  -> 验证损失未改善，已持续 {counter} 个epoch。')
                if counter >= patience:
                    print(f"  -> 早停触发！在第 {epoch+1} 个epoch停止训练。")
                    break
        
        # 一组参数训练结束后，评估其性能
        print(f"\n第 {combo_idx} 组训练结束。最佳验证损失: {best_val_loss:.6f}")
        
        # 更新全局最佳
        if best_val_loss < global_best_val_loss:
            global_best_val_loss = best_val_loss
            global_best_hyperparams = {
                'lr': lr,
                'batch_size': batch_size,
                'weight_decay': weight_decay,
                'T_0': T_0,
                'T_mult': T_mult,
                'eta_min': eta_min,
                'best_val_loss': best_val_loss
            }
            global_best_model_state = best_model_state
            print(f"*** 发现新的全局最佳模型! ***")

    # 所有组合训练完毕后，进行最终测试 
    print("\n" + "="*60)
    print("所有超参数组合训练完毕!")
    print("="*60)
    
    if global_best_model_state is not None:
        print("\n全局最佳超参数组合:")
        for key, value in global_best_hyperparams.items():
            print(f"  {key}: {value}")
        
        print(f"\n使用全局最佳模型在测试集上进行评估...")
        final_test_model = CodecSystem()
        final_test_model.load_state_dict(global_best_model_state)
        final_test_model.to(device)
        final_test_model.eval()
        
        total_test_loss = 0.0
        with torch.no_grad():
            for batch_data in test_loader:
                batch_data = batch_data.to(device)
                outputs, _, _ = final_test_model(batch_data)
                loss = criterion(outputs, batch_data)
                total_test_loss += loss.item() * batch_data.size(0)
        
        average_test_loss = total_test_loss / len(test_dataset)
        print(f"\n全局最佳模型的测试集平均L1 Loss: {average_test_loss:.6f}")
    else:
        print("未找到有效的训练结果。")
